# Swept Pazy wing: flutter onset sensitivity to structural mass and stiffness uncertainty

The Technion swept Pazy wing structural model is represented with discrete lumped point
masses and cross-sectional stiffness properties. Each lumped mass is characterised by its measured
mass, position offset vector, and rotational inertia tensor. Each element stiffness contains a partially-populated
`(6, 6)` cross-sectional stiffness matrix; this notebook investigates the sensitivity of the
flutter onset velocity to uncertainty in the lumped-mass properties and the torsional, out-of-plane, and in-plane
bending stiffness terms (`k22, k33, k44`).

The approach taken is to first identify the flutter onset velocity. From this, we can use the adjoint
routine for the linear system to get the sensitivities of the real part of the flutter eigenvalue with respect
to all aforementioned structural properties. Note that the flutter onset velocity is not an explicit
function of the structural properties. This notebook runs the adjoint solve, and compares with a Monte Carlo verifification.

## Imports

In [ ]:
from collections.abc import Callable
from copy import deepcopy
from functools import partial

import jax
import numpy as np
from jax import Array
from jax import numpy as jnp
from matplotlib import pyplot as plt

from flapjax.aero.flowfields import ConstantFlowField
from flapjax.aero.gradients.data_structures import AeroGradsToCompute
from flapjax.coupled import (
    AeroelasticGradsToCompute,
    CoupledAeroelastic,
    stability_adjoint,
)
from flapjax.models.pazy.base import (
    build_lumped_mass_matrices,
    build_stiffness_matrices,
)
from flapjax.models.pazy.swept.data.properties_10_deg import SWEEP_10_TE
from flapjax.models.pazy.swept.swept_pazy_wing import generate_swept_pazy_wing
from flapjax.structure.gradients.data_structures import StructureGradsToCompute
from flapjax.utils.print_utils import set_verbosity

set_verbosity("silent")

## Discretisation and wing setup

We use the 10 degree swept Pazy wing with the leading-edge, frequency-corrected tip mass.

The aero resolution (`M`, `M_STAR`) matches the configuration already validated for
`stability_adjoint` in `tests/coupled/static/test_stability_adjoint.py`.

In [ ]:
M = 8  # wing chordwise discretisation
M_STAR = 40  # number of wake panels per strip
AOA = jnp.deg2rad(5.0)  # angle of attack
RHO = 1.225
DATA = SWEEP_10_TE  # keypoint mass/CG/inertia/stiffness data
N_MASS = DATA.mass.shape[0]
N_ELEM = N_MASS - 1

MASS0 = DATA.mass  # (N_MASS,)
CG0 = jnp.stack((DATA.cg_x, DATA.cg_y, DATA.cg_z), axis=1)  # (N_MASS, 3)
INERTIA0 = jnp.stack((DATA.i_xx, DATA.i_yy, DATA.i_zz, DATA.i_xy, DATA.i_xz, DATA.i_yz), axis=1)  # (N_MASS, 6)
CG_FIELDS = ("cg_x", "cg_y", "cg_z")
INERTIA_FIELDS = ("i_xx", "i_yy", "i_zz", "i_xy", "i_xz", "i_yz")
STIFFNESS_FIELDS = ("k22", "k33", "k44")

CG_LABELS = ("$r_x$", "$r_y$", "$r_z$")
INERTIA_LABELS = ("$I_{xx}$", "$I_{yy}$", "$I_{zz}$", "$I_{xy}$", "$I_{xz}$", "$I_{yz}$")
STIFFNESS_LABELS = ("$GJ$", "$EIy$", "$EIz$")

M_LUMPED0 = build_lumped_mass_matrices(DATA, mass=MASS0, cg=CG0, inertia=INERTIA0)

K_CS_PARAMS = {name: getattr(DATA, name) for name in STIFFNESS_FIELDS}  # each (N_ELEM,)


def k_cs_of_params(k22: Array, k33: Array, k44: Array) -> Array:
    # the wing below uses node_multiplier=1, so the per-segment data already sits at the beam's
    # final per-element midpoints and needs no interpolation onto a different discretisation.
    return build_stiffness_matrices(DATA, k22=k22, k33=k33, k44=k44)


K_CS0 = k_cs_of_params(**K_CS_PARAMS)

base_wing: CoupledAeroelastic = generate_swept_pazy_wing(
    m=M,
    m_star=M_STAR,
    sweep_angle=10,
    tip_mass="LE_CORRECTED",
    node_multiplier=1,
    aoa=AOA,
    lumped_mass=True,
    flowfield=ConstantFlowField(
        u_inf=jnp.array((50.0, 0.0, 0.0)),  # overwritten per velocity below
        rho=RHO,
        relative_motion=True,
    ),
)


def apply_dv(
        wing: CoupledAeroelastic, u_mag: float | Array, m_lumped: Array, k_cs: Array,
        dt: float | Array | None = None,
) -> None:
    """Set the design variables for a given freestream speed, lumped-mass matrix array, and per-element
    stiffness matrix array.

    `dt` can be pinned to a fixed value (rather than recomputed from `u_mag`) so that
    a velocity-only finite-difference comparison does not also perturb the time step.
    """
    dt_ = dt if dt is not None else 0.1 / (M * u_mag)
    wing.set_design_variables(
        coords=wing.structure.x0,
        k_cs=k_cs,
        m_cs=wing.structure.m_cs,
        m_lumped=m_lumped,
        dt=dt_,
        flowfield=ConstantFlowField(u_inf=jnp.array([u_mag, 0.0, 0.0]), rho=RHO, relative_motion=True),
        zeta_b0=wing.aero.zeta_b0,
        remove_checks=True,
    )

## Locate the nominal flutter onset velocity

Locate flutter with a bisection method, which minimises the number of linearisations and eigendecompositions needed to find the onset velocity.


In [ ]:
def bisect_flutter_onset(
        eigs_at_velocity_: Callable[[float], Array],
        u_lo: float,
        u_hi: float,
        tol: float = 0.05,
        max_iter: int = 30,
) -> float:
    """
    Locate the velocity at which the continuous-time eigenvalues cross from stable to unstable, by bisection.
    """

    def min_zeta(u: float) -> float:
        evals = eigs_at_velocity_(u)
        zeta = -evals.real / jnp.abs(evals)
        return float(zeta.min())

    z_lo = min_zeta(u_lo)
    z_hi = min_zeta(u_hi)
    if not (z_lo > 0.0 > z_hi):
        raise ValueError(
            f"Bracket [{u_lo}, {u_hi}] does not bracket a stable-to-unstable crossing: "
            f"minimum damping ratio is {z_lo:.4f} at u_lo and {z_hi:.4f} at u_hi."
        )

    for _ in range(max_iter):
        if (u_hi - u_lo) < tol:
            break
        u_mid = 0.5 * (u_lo + u_hi)
        z_mid = min_zeta(u_mid)
        if z_mid > 0.0:
            u_lo, z_lo = u_mid, z_mid
        else:
            u_hi, z_hi = u_mid, z_mid

    return u_lo - z_lo / (z_hi - z_lo) * (u_hi - u_lo)


def _static_solve_and_eigs(wing: CoupledAeroelastic, u_mag: Array, m_lumped: Array, k_cs: Array) -> Array:
    dt = 0.1 / (M * u_mag)
    wing.set_design_variables(
        coords=wing.structure.x0,
        k_cs=k_cs,
        m_cs=wing.structure.m_cs,
        m_lumped=m_lumped,
        dt=dt,
        flowfield=ConstantFlowField(u_inf=jnp.array([u_mag, 0.0, 0.0]), rho=RHO, relative_motion=True),
        zeta_b0=wing.aero.zeta_b0,
        remove_checks=True,
    )
    static_sol = wing.static_solve(prescribed_dofs=tuple(range(6)), horseshoe=False)
    linear_sol = wing.linearise(reference=static_sol, skip_checks=True, batch_size=4, n_struct_modes=20)
    return linear_sol.modal(n_modes=5, plot_eigvals=False, freq_range=(0.0, 50.0), damp_range=(-jnp.inf, 0.3))


# compiled on the first call below, then the compiled function is reused for efficiency
_eigs_at_velocity_jit = jax.jit(_static_solve_and_eigs)


def eigs_at_velocity(u_mag: Array, m_lumped: Array, k_cs: Array, wing: CoupledAeroelastic) -> Array:
    return _eigs_at_velocity_jit(wing, u_mag, m_lumped, k_cs)


def find_onset(
        u_lo: float, u_hi: float, m_lumped: Array, k_cs: Array, wing: CoupledAeroelastic, tol: float = 0.05
) -> float:
    """Locate the flutter onset velocity within [u_lo, u_hi]."""

    return bisect_flutter_onset(
        partial(eigs_at_velocity, m_lumped=m_lumped, k_cs=k_cs, wing=wing), u_lo=u_lo, u_hi=u_hi, tol=tol
    )


u_f0 = find_onset(35.0, 85.0, M_LUMPED0, K_CS0, base_wing)

## Identify the flutter mode

At the flutter velocity, the mode's damping is approximately zero. However, in such a model, where may be many very high frequency aerodynamic instabilities which are not physical. The adjoint objective below only sees the raw continuous-time eigenvalues, not the eigenvectors, so we need a robust way to identify the actual flutter mode. We do this by finding the mode closest to neutral stability, subject to a frequency band constraint that excludes high-frequency numerical artefacts. For subsequent evaluations, we find the flutter mode as the eigenvalue nearest the reference value found at the nominal flutter velocity.

In [ ]:
def full_order_eigs(
        u_mag: float | Array, m_lumped: Array, k_cs: Array, wing: CoupledAeroelastic,
        dt: float | Array | None = None,
) -> Array:
    inner_wing = deepcopy(wing)
    dt_ = dt if dt is not None else 0.1 / (M * u_mag)
    apply_dv(inner_wing, u_mag, m_lumped, k_cs, dt=dt_)
    static_sol = inner_wing.static_solve(prescribed_dofs=tuple(range(6)), horseshoe=False)
    linear_sol = inner_wing.linearise(reference=static_sol, skip_checks=True, batch_size=4, n_struct_modes=None)
    evals_d = jnp.linalg.eig(linear_sol.sys.a)[0]
    return jnp.log(evals_d) / dt_


def bootstrap_critical_mode(
        u_mag: float | Array, m_lumped: Array, k_cs: Array, wing: CoupledAeroelastic,
        freq_band: tuple[float, float] = (1.0, 50.0),
) -> Array:
    evals_c = full_order_eigs(u_mag, m_lumped, k_cs, wing)
    freq_hz = jnp.abs(evals_c.imag) / (2.0 * jnp.pi)
    in_band = (freq_hz > freq_band[0]) & (freq_hz < freq_band[1])
    abs_real_masked = jnp.where(in_band, jnp.abs(evals_c.real), jnp.inf)
    return evals_c[jnp.argmin(abs_real_masked)]


LAMBDA_REF = bootstrap_critical_mode(u_f0, M_LUMPED0, K_CS0, base_wing)


def critical_eigenvalue(
        u_mag: float | Array, m_lumped: Array, k_cs: Array, wing: CoupledAeroelastic,
        dt: float | Array | None = None,
) -> Array:
    """Continuous-time eigenvalue nearest `LAMBDA_REF`, tracking the critical mode under perturbation."""
    evals_c = full_order_eigs(u_mag, m_lumped, k_cs, wing, dt=dt)
    return evals_c[jnp.argmin(jnp.abs(evals_c - LAMBDA_REF))]


## Adjoint sensitivity of the flutter margin to each lumped mass and stiffness matrix

Build the static equilibrium at the flutter velocity and call `stability_adjoint` to get the
gradient of the critical mode's real part with respect to the structural properties of interest.

In [ ]:
wing_uf = deepcopy(base_wing)
apply_dv(wing_uf, u_f0, M_LUMPED0, K_CS0)
case_uf = wing_uf.static_solve(prescribed_dofs=tuple(range(6)), horseshoe=False)


def flutter_margin(states, dv, lambda_c: Array) -> Array:
    """Real part of the continuous-time eigenvalue nearest the flutter mode."""
    del states, dv
    idx = jnp.argmin(jnp.abs(lambda_c - LAMBDA_REF))
    return jnp.real(lambda_c[idx])


grads_to_compute = AeroelasticGradsToCompute(
    structure=StructureGradsToCompute(k_cs=True, m_cs=False, m_lumped=True),
    aero=AeroGradsToCompute(zeta_b0=False, flowfield=True),
)

g_val, dv_bar = stability_adjoint(
    system=wing_uf, case=case_uf, objective=flutter_margin, grads_to_compute=grads_to_compute
)

dg_dm_lumped = dv_bar.structure.m_lumped  # (N_MASS, 6, 6)
dg_dk_cs = dv_bar.structure.k_cs  # (N_ELEM, 6, 6)


## Velocity sensitivity of the flutter margin

By the implicit function theorem, since `g(U_f, params) = 0` defines the flutter
onset velocity as a function of the mass/stiffness parameters:

$$\frac{dU_f}{d(\cdot)} = -\frac{\partial g / \partial(\cdot)}{\partial g / \partial U}$$

`stability_adjoint` can also return `∂g/∂U` (as the gradient with respect to the
`u_inf` flowfield design variable), reusing the same adjoint solve above. Since `∂g/∂U` is a
single scalar, a direct finite difference is trivial to compute.

In [ ]:
dt_fixed = 0.1 / (M * u_f0)
h_dgdu = 0.1
g_plus = critical_eigenvalue(u_f0 + h_dgdu, M_LUMPED0, K_CS0, base_wing, dt=dt_fixed).real
g_minus = critical_eigenvalue(u_f0 - h_dgdu, M_LUMPED0, K_CS0, base_wing, dt=dt_fixed).real
dg_dU = float((g_plus - g_minus) / (2.0 * h_dgdu))


## Obtain gradient for physical inputs

`dg_dm_lumped` and `dg_dk_cs` are the gradients of the flutter margin with respect to
every entry of every `(6, 6)` lumped mass matrix and every `(6, 6)` element stiffness
matrix respectively. We can go from sensitivities with respect to the full matrices to sensitivities
with respect to the smaller space of physically-meaningful parameters.

In [ ]:
def m_lumped_of_params(mass: Array, cg: Array, inertia: Array) -> Array:
    return build_lumped_mass_matrices(DATA, mass=mass, cg=cg, inertia=inertia)


_, vjp_mass_params = jax.vjp(m_lumped_of_params, MASS0, CG0, INERTIA0)
dg_dmass, dg_dcg, dg_dinertia = vjp_mass_params(dg_dm_lumped)

_, vjp_stiffness_params = jax.vjp(k_cs_of_params, *(K_CS_PARAMS[name] for name in STIFFNESS_FIELDS))
dg_dstiffness = dict(zip(STIFFNESS_FIELDS, vjp_stiffness_params(dg_dk_cs)))

dVf_dmass = -dg_dmass / dg_dU  # (N_MASS,)       d(U_f) / d(mass_i)
dVf_dcg = -dg_dcg / dg_dU  # (N_MASS, 3)         d(U_f) / d(cg_i, axis)
dVf_dinertia = -dg_dinertia / dg_dU  # (N_MASS, 6)   d(U_f) / d(inertia_i, component)
dVf_dstiffness = {name: -grad / dg_dU for name, grad in dg_dstiffness.items()}  # (N_ELEM,) each

## Define parameter uncertainty

Every physical quantity is given it's own standard deviation. These are given as a coefficient of variation for quantities that have a nonzero nominal value, and as an absolute uncertainty for quantities that can be close to zero. The coefficient of variation is applied to the mean nominal value across all entries of that variable, not each entry's own value, so every entry of a given type (e.g. every node's `i_yy`) shares the same absolute standard deviation rather than larger entries being assigned a proportionally larger variance. These are arbitrarily chosen, and can be replaced with actual measured/identified tolerances if available.

In [ ]:
VARIABLES: dict[str, Array] = {
    "mass": MASS0,
    "cg_x": CG0[:, 0],
    "cg_y": CG0[:, 1],
    "cg_z": CG0[:, 2],
    "i_xx": INERTIA0[:, 0],
    "i_yy": INERTIA0[:, 1],
    "i_zz": INERTIA0[:, 2],
    "i_xy": INERTIA0[:, 3],
    "i_xz": INERTIA0[:, 4],
    "i_yz": INERTIA0[:, 5],
    **{name: K_CS_PARAMS[name] for name in STIFFNESS_FIELDS},
}

SENSITIVITIES: dict[str, Array] = {
    "mass": dVf_dmass,
    "cg_x": dVf_dcg[:, 0],
    "cg_y": dVf_dcg[:, 1],
    "cg_z": dVf_dcg[:, 2],
    "i_xx": dVf_dinertia[:, 0],
    "i_yy": dVf_dinertia[:, 1],
    "i_zz": dVf_dinertia[:, 2],
    "i_xy": dVf_dinertia[:, 3],
    "i_xz": dVf_dinertia[:, 4],
    "i_yz": dVf_dinertia[:, 5],
    **{name: dVf_dstiffness[name] for name in STIFFNESS_FIELDS},
}

UNCERTAINTY_SPEC: dict[str, tuple[str, float]] = {
    "mass": ("cov", 0.02),  # 2% standard deviation for lumped masses
    "cg_x": ("abs", 0.5e-3),  # 0.5 mm standard deviation for mass position
    "cg_y": ("abs", 0.5e-3),
    "cg_z": ("abs", 0.5e-3),
    "i_xx": ("cov", 0.05),  # 5% standard deviation for inertia tensor components
    "i_yy": ("cov", 0.05),
    "i_zz": ("cov", 0.05),
    "i_xy": ("cov", 0.05),
    "i_xz": ("cov", 0.05),
    "i_yz": ("cov", 0.05),
    "k22": ("cov", 0.05),  # 5% standard deviation for element stiffnesses
    "k33": ("cov", 0.05),
    "k44": ("cov", 0.05),
}


def sigma_of(name: str, nominal: Array) -> Array:
    """Standard deviation for every entry of `nominal`."""
    mode, value = UNCERTAINTY_SPEC[name]
    if mode == "cov":
        return jnp.full_like(nominal, value * jnp.mean(jnp.abs(nominal)))
    if mode == "abs":
        return jnp.full_like(nominal, value)
    raise ValueError(f"Unknown uncertainty mode {mode!r} for variable {name!r}")


sigma = {name: sigma_of(name, nominal) for name, nominal in VARIABLES.items()}
var_contrib = {name: (SENSITIVITIES[name] * sigma[name]) ** 2 for name in VARIABLES}

var_Uf = float(sum(jnp.sum(v) for v in var_contrib.values()))
std_Uf = float(jnp.sqrt(var_Uf))

CATEGORIES: dict[str, tuple[str, ...]] = {
    "Mass magnitude": ("mass",),
    "Mass position": CG_FIELDS,
    "Mass rotational inertia": INERTIA_FIELDS,
    "Torsional stiffness": ("k22",),
    "Out-of-plane stiffness": ("k33",),
    "In-plane stiffness": ("k44",),
}
CATEGORY_VAR: dict[str, float] = {
    cat_name: sum(float(jnp.sum(var_contrib[name])) for name in names) for cat_name, names in CATEGORIES.items()
}

## Sensitivity ranking

Find the top contributors to uncertainty in the flutter onset velocity.

In [ ]:
entries = []
for i in range(N_MASS):
    entries.append((f"Node {i} mass", float(jnp.sqrt(var_contrib["mass"][i]))))
    for field, label in zip(CG_FIELDS, CG_LABELS):
        entries.append((f"Node {i} {label}", float(jnp.sqrt(var_contrib[field][i]))))
    for field, label in zip(INERTIA_FIELDS, INERTIA_LABELS):
        entries.append((f"Node {i} {label}", float(jnp.sqrt(var_contrib[field][i]))))
for i in range(N_ELEM):
    for field, label in zip(STIFFNESS_FIELDS, STIFFNESS_LABELS):
        entries.append((f"Element {i}, {label}", float(jnp.sqrt(var_contrib[field][i]))))

entries.sort(key=lambda e: e[1], reverse=True)
top_n = 15
labels_top = [e[0] for e in entries[:top_n]][::-1]
values_top = [e[1] for e in entries[:top_n]][::-1]

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.barh(labels_top, values_top, color="tab:blue")
ax.set_xlabel("Flutter speed shift from this entry's 1-sigma uncertainty, m/s")
ax.set_title(f"Top {top_n} contributors to flutter speed uncertainty")
plt.tight_layout()
plt.show()

### Variance Split

The ranking above is for individual entries; below, we group these.

In [ ]:
cat_names_sorted, cat_var_sorted = zip(*sorted(CATEGORY_VAR.items(), key=lambda kv: kv[1], reverse=True))
cat_pct_sorted = [100.0 * v / var_Uf for v in cat_var_sorted]
fig, ax = plt.subplots(figsize=(6.5, 4.5))
bars = ax.bar(cat_names_sorted, cat_pct_sorted, color="tab:blue")
ax.set_ylabel("Percentage of total flutter-speed variance")
ax.set_title("Contribution of each parameter category to flutter speed uncertainty")
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
plt.tight_layout()
plt.show()

### Spanwise sensitivity breakdown

The plots below show the raw sensitivities for all structural parameters, plotted at spanwise stations.

In [ ]:
y_span = np.array(DATA.coords[:, 1])
y_span_elem = np.array(0.5 * (DATA.coords[:-1, 1] + DATA.coords[1:, 1]))

fig, ((ax_mass, ax_cg), (ax_inertia, ax_stiffness)) = plt.subplots(2, 2, figsize=(10, 8), sharex=True)

ax_mass.plot(y_span, dVf_dmass, marker="o", color="tab:blue")
ax_mass.set_title("Mass")
ax_mass.set_ylabel(r"$\partial~U_f/\partial~m$")

for i, (label, color) in enumerate(zip(CG_LABELS, ("tab:blue", "tab:orange", "tab:green"))):
    ax_cg.plot(y_span, dVf_dcg[:, i], marker="o", color=color, label=label)
ax_cg.set_title("Mass Offset")
ax_cg.set_ylabel(r"$\partial~U_f/\partial~\mathbf{r}$")
ax_cg.legend(fontsize=8)

for i, label in enumerate(INERTIA_LABELS):
    ax_inertia.plot(y_span, dVf_dinertia[:, i], marker="o", label=label)
ax_inertia.set_title("Rotational Inertia")
ax_inertia.set_ylabel(r"$\partial~U_f/\partial~\mathbf{I}$")
ax_inertia.legend(fontsize=8, ncol=2)

for i, (field, label) in enumerate(zip(STIFFNESS_FIELDS, STIFFNESS_LABELS)):
    ax_stiffness.plot(y_span_elem, SENSITIVITIES[field], marker="o", label=label)
ax_stiffness.set_title("Stiffness")
ax_stiffness.set_ylabel(r"$\partial~U_f/\partial~\mathbf{K}_{cs}$")
ax_stiffness.legend(fontsize=7, ncol=2)

for ax in (ax_mass, ax_cg, ax_inertia, ax_stiffness):
    ax.axhline(0.0, color="k", linewidth=0.5)
for ax in (ax_inertia, ax_stiffness):
    ax.set_xlabel("Spanwise position, y [m]")

fig.suptitle("Flutter onset velocity $U_f$ sensitivity to each parameter")
plt.tight_layout()
plt.show()

## Monte Carlo comparison
We can reconstruct a Gaussian distribution by randomly sampling all of the input parameters according to their variance, and running nonlinear analysis, to give an output distribution of flutter onset velocity.

In [ ]:
N_MC = 600
rng = np.random.default_rng(0)
half_width = max(1.5, 6.0 * std_Uf)


def sample_variable(name: str) -> Array:
    """Draw one Monte Carlo sample of `VARIABLES[name]`, perturbed by `sigma[name]`"""
    nominal = VARIABLES[name]
    noise = rng.standard_normal(nominal.shape)
    return nominal + sigma[name] * noise


mc_samples = []
for i_mc in range(N_MC):
    mass_sample = sample_variable("mass")
    cg_sample = jnp.stack([sample_variable(name) for name in CG_FIELDS], axis=1)
    inertia_sample = jnp.stack([sample_variable(name) for name in INERTIA_FIELDS], axis=1)
    m_lumped_sample = build_lumped_mass_matrices(DATA, mass=mass_sample, cg=cg_sample, inertia=inertia_sample)

    stiffness_sample = {name: sample_variable(name) for name in STIFFNESS_FIELDS}
    k_cs_sample = k_cs_of_params(**stiffness_sample)

    u_mc = find_onset(u_f0 - half_width, u_f0 + half_width, m_lumped_sample, k_cs_sample, base_wing)
    if u_mc is not None:
        mc_samples.append(u_mc)
    print(f"  sample {i_mc + 1:2d}/{N_MC}: U_f = {u_mc}")

mc_samples = np.array(mc_samples)
mc_mean, mc_std = mc_samples.mean(), mc_samples.std(ddof=1)
print(f"\nMonte Carlo ({mc_samples.size} samples): mean={mc_mean:.3f} m/s, std={mc_std:.3f} m/s")
print(f"FOSM prediction:                    mean={u_f0:.3f} m/s, std={std_Uf:.3f} m/s")

In [ ]:
u_axis = jnp.linspace(u_f0 - half_width, u_f0 + half_width, 200)
gaussian_pdf = jnp.exp(-0.5 * ((u_axis - u_f0) / std_Uf) ** 2) / (std_Uf * jnp.sqrt(2.0 * jnp.pi))

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.hist(mc_samples, bins=max(5, N_MC // 8), density=True, color="tab:blue", alpha=0.6, label="Monte Carlo samples")
ax.plot(np.array(u_axis), np.array(gaussian_pdf), color="tab:red", label="FOSM Gaussian approximation")
ax.axvline(u_f0, color="k", linewidth=0.8, linestyle="--", label=f"U_f0 = {u_f0:.2f} m/s")
ax.set_xlabel("Flutter onset velocity [m/s]")
ax.set_ylabel("Probability density")
ax.set_title("Flutter onset velocity under structural mass and stiffness uncertainty")
ax.legend(fontsize=8)
ax.set_xlim(42.0, 50.0)
plt.tight_layout()
plt.show()
